<a href="https://colab.research.google.com/github/behan02/langchain-tutorial/blob/main/chapters/02-langsmith.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Langsmith introduction

LangSmith is a built-in observability service and platform that integrates very easily with LangChain.

In [ ]:
!pip install \
  langchain-core \
  langchain-google-genai \
  langchain-community \
  langsmith

#Setting up LangSmith

LangSmith does require an API key, but it comes with a generous free tier.

When using LangSmith, we need to setup our environment variables and provide our API key.

In [2]:
import os

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGCHAIN_API_KEY"] = ""
os.environ["LANGCHAIN_PROJECT"] = ""

#Default Tracing

As mentioned, LangSmith traces a lot of data without us needing to do anything. Let's see how that looks. We'll start by initializing our LLM. Again, this will need an API key.

In [3]:
from langchain_google_genai import ChatGoogleGenerativeAI

os.environ["GOOGLE_API_KEY"] = ""

llm = ChatGoogleGenerativeAI(temperature=0.0, model="gemini-2.5-flash")

In [ ]:
llm.invoke("Hello")

#Tracing Non-LangChain Code

LangSmith can trace functions that are not part of LangChain, we just need to add the @traceable decorator. Let's try this for a few simple functions.

In [5]:
from langsmith import traceable
import random
import time


@traceable
def generate_random_number():
    return random.randint(0, 100)

@traceable
def generate_string_delay(input_str: str):
    number = random.randint(1, 5)
    time.sleep(number)
    return f"{input_str} ({number})"

@traceable
def random_error():
    number = random.randint(0, 1)
    if number == 0:
        raise ValueError("Random error")
    else:
        return "No error"

In [ ]:
from tqdm.auto import tqdm

for _ in tqdm(range(10)):
    generate_random_number()
    generate_string_delay("Hello")
    try:
        random_error()
    except ValueError:
        pass

Finally, we can also modify our traceable names if we'd like to make them more readable inside the UI. For example:

In [7]:
from langsmith import traceable

@traceable(name="Chitchat Maker")
def error_generation_function(question: str):
    delay = random.randint(0, 3)
    time.sleep(delay)
    number = random.randint(0, 1)
    if number == 0:
        raise ValueError("Random error")
    else:
        return "I'm great how are you?"

In [ ]:
for _ in tqdm(range(10)):
    try:
        error_generation_function("How are you today?")
    except ValueError:
        pass